# 串行中断
## 跨节点串行中断

In [18]:
from typing import TypedDict
from rich import print
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt


# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点：串行路径上每个节点各带一个 interrupt
def ask_name(state: State) -> dict:
    print("========= 执行 ask_name 节点 =========")
    username = interrupt("请输入你的名字")
    return {"username": username}


def ask_age(state: State) -> dict:
    print("========= 执行 ask_age 节点 =========")
    age = interrupt("请输入你的年龄")
    return {"age": age}


# 构建图结构：ask_name -> ask_age -> summarize 串行链
builder = StateGraph(state_schema=State)
builder.add_node("ask_name", ask_name)
builder.add_node("ask_age", ask_age)
builder.add_edge(START, "ask_name")
builder.add_edge("ask_name", "ask_age")
builder.add_edge("ask_age", END)

# 设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

========= 执行 ask_name 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的名字', id='ab723af8810a27469a57489bdea873ed')]}

In [19]:
while res.get("__interrupt__"):
    ask_msg = res["__interrupt__"][0].value
    answer = input(ask_msg)
    res = graph.invoke(Command(resume=answer), config=config)
    print(res)

========= 执行 ask_name 节点 =========

========= 执行 ask_age 节点 =========

{
    'username': 'aihaipeng',
    '__interrupt__': [Interrupt(value='请输入你的年龄', id='e8337d42809d6c7791144d23039a2fca')]
}

========= 执行 ask_age 节点 =========

{'username': 'aihaipeng', 'age': '26'}

In [20]:
print(list(graph.get_state_history(config=config)))

[
    StateSnapshot(
        values={'username': 'aihaipeng', 'age': '26'},
        next=(),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-de27-6591-8002-7d515ea2df47'
            }
        },
        metadata={'source': 'loop', 'step': 2, 'parents': {}},
        created_at='2026-09-13T17:41:36.014265+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-c74e-6108-8001-6a61cca34a0b'
            }
        },
        tasks=(),
        interrupts=()
    ),
    StateSnapshot(
        values={'username': 'aihaipeng'},
        next=('ask_age',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-c74e-6108-8001-6a61cca34a0b'
            }
        },
        metadata={'source': 'loop', 'step': 1, 'parents': {}},
        created_at='2026-09-13T17:41:33.618399+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-87f6-6f55-8000-56971b730d9d'
            }
        },
        tasks=(
            PregelTask(
                id='fb4dd0e9-bf1f-31dd-001b-538c05953a05',
                name='ask_age',
                path=('__pregel_pull', 'ask_age'),
                error=None,
                interrupts=(Interrupt(value='请输入你的年龄', id='e8337d42809d6c7791144d23039a2fca'),),
                state=None,
                result={'age': '26'}
            ),
        ),
        interrupts=(Interrupt(value='请输入你的年龄', id='e8337d42809d6c7791144d23039a2fca'),)
    ),
    StateSnapshot(
        values={},
        next=('ask_name',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-87f6-6f55-8000-56971b730d9d'
            }
        },
        metadata={'source': 'loop', 'step': 0, 'parents': {}},
        created_at='2026-09-13T17:41:26.976693+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-87f5-6c34-bfff-b940d194e436'
            }
        },
        tasks=(
            PregelTask(
                id='479d4c27-4bf0-5300-e3b6-209f54d35f6d',
                name='ask_name',
                path=('__pregel_pull', 'ask_name'),
                error=None,
                interrupts=(Interrupt(value='请输入你的名字', id='ab723af8810a27469a57489bdea873ed'),),
                state=None,
                result={'username': 'aihaipeng'}
            ),
        ),
        interrupts=(Interrupt(value='请输入你的名字', id='ab723af8810a27469a57489bdea873ed'),)
    ),
    StateSnapshot(
        values={},
        next=('__start__',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9a5-87f5-6c34-bfff-b940d194e436'
            }
        },
        metadata={'source': 'input', 'step': -1, 'parents': {}},
        created_at='2026-09-13T17:41:26.976206+00:00',
        parent_config=None,
        tasks=(
            PregelTask(
                id='e58ae24f-7b3e-d3cf-3702-7ff86ac57cfd',
                name='__start__',
                path=('__pregel_pull', '__start__'),
                error=None,
                interrupts=(),
                state=None,
                result={}
            ),
        ),
        interrupts=()
    )
]

## 单节点串行中断

In [21]:
from typing import TypedDict
from rich import print
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt


# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点
def ask_user_info(state: State) -> dict:
    print("========= 执行 ask_user_info 节点 =========")
    username = interrupt("请输入你的名字")
    age = interrupt("请输入你的年龄")
    return {
        "username": username,
        "age": age
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("ask_user_info", ask_user_info)
builder.add_edge(START, "ask_user_info")
builder.add_edge("ask_user_info", END)

# 设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

========= 执行 ask_user_info 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的名字', id='d1b4e62f2c2c56bd661d069491bc451f')]}

In [22]:
while res.get("__interrupt__"):
    ask_msg = res["__interrupt__"][0].value
    answer = input(ask_msg)
    res = graph.invoke(Command(resume=answer), config=config)
    print(res)

========= 执行 ask_user_info 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的年龄', id='d1b4e62f2c2c56bd661d069491bc451f')]}

========= 执行 ask_user_info 节点 =========

{'username': 'aihaipeng', 'age': '26'}

In [23]:
print(list(graph.get_state_history(config=config)))

[
    StateSnapshot(
        values={'username': 'aihaipeng', 'age': '26'},
        next=(),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9ae-6937-651e-8001-c72447a7de2b'
            }
        },
        metadata={'source': 'loop', 'step': 1, 'parents': {}},
        created_at='2026-09-13T17:45:25.344379+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9ae-03d1-60ed-8000-ed721d0455c6'
            }
        },
        tasks=(),
        interrupts=()
    ),
    StateSnapshot(
        values={},
        next=('ask_user_info',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9ae-03d1-60ed-8000-ed721d0455c6'
            }
        },
        metadata={'source': 'loop', 'step': 0, 'parents': {}},
        created_at='2026-09-13T17:45:14.711877+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9ae-03cf-6c20-bfff-9db8c79e915f'
            }
        },
        tasks=(
            PregelTask(
                id='4c872ef9-f87d-099e-cfd7-d5c11a771370',
                name='ask_user_info',
                path=('__pregel_pull', 'ask_user_info'),
                error=None,
                interrupts=(Interrupt(value='请输入你的年龄', id='d1b4e62f2c2c56bd661d069491bc451f'),),
                state=None,
                result={'username': 'aihaipeng', 'age': '26'}
            ),
        ),
        interrupts=(Interrupt(value='请输入你的年龄', id='d1b4e62f2c2c56bd661d069491bc451f'),)
    ),
    StateSnapshot(
        values={},
        next=('__start__',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1af9ae-03cf-6c20-bfff-9db8c79e915f'
            }
        },
        metadata={'source': 'input', 'step': -1, 'parents': {}},
        created_at='2026-09-13T17:45:14.711347+00:00',
        parent_config=None,
        tasks=(
            PregelTask(
                id='aecab823-51c5-9f45-681c-a822ce2ee742',
                name='__start__',
                path=('__pregel_pull', '__start__'),
                error=None,
                interrupts=(),
                state=None,
                result={}
            ),
        ),
        interrupts=()
    )
]